# Build core financial dataset

This notebook transforms the financial statement staging data into a standardized club-year dataset for analysis.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel('../data/staging/financial_data_staging.xlsx')
df_copy = df.copy()

## Normalize monetary units

The source statements use both BRL and BRL thousands. Values are converted to BRL thousands while keeping the original values unchanged.

In [2]:
df_copy['unit'].value_counts()

unit
BRL_thousands    1204
BRL              1176
Name: count, dtype: int64

In [3]:
df_copy['value_brl_thousands'] = np.where(
    df_copy['unit'] == 'BRL',
    df_copy['value_original'] / 1000,
    df_copy['value_original']
)

## Reshape the data

Convert the staging data from long format to one row per club and fiscal year.

In [4]:
# sum is needed for variables built from multiple accounting components
core = (
    df_copy.pivot_table(
    values='value_brl_thousands',
    index=['club', 'fiscal_year'],
    columns='standard_variable',
    aggfunc='sum'
).rename_axis(None, axis='columns')
.reset_index()
)

## Derive missing core variables

Some clubs do not report all core variables directly. Missing values are reconstructed from the accounting components collected during the staging process.

In [5]:
core['current_assets'] = np.where(
    core['current_assets'].isna(),
    core['total_assets'] - core['noncurrent_assets_component'],
    core['current_assets']
)

core['total_liabilities'] = np.where(
    core['total_liabilities'].isna(),
    core['current_liabilities'].fillna(0) + 
    core['noncurrent_liabilities'].fillna(0) + 
    core['future_results'].fillna(0),
    core['total_liabilities']
)

## Check data quality

Check if the new data has missing values, incomplete years or accounting inconsistencies before creating the final dataset.

In [6]:
core_variables = ['cash', 'current_assets', 'total_assets', 'current_liabilities', 
'total_liabilities', 'equity', 'net_result']

core[core[core_variables].isna().any(axis=1)]

,club,fiscal_year,cash,current_assets,current_liabilities,equity,future_results,net_result,noncurrent_assets_component,noncurrent_liabilities,total_assets,total_liabilities


In [7]:
core.groupby('club')['fiscal_year'].agg(['min', 'max', 'count'])

,min,max,count
club,,,
América-MG,2016,2025,10
Athletico-PR,2016,2025,10
Atlético-GO,2016,2025,10
Atlético-MG,2016,2025,10
Avaí,2016,2025,10
Bahia,2016,2025,10
Botafogo,2016,2025,10
CSA,2016,2024,9
Ceará,2016,2025,10


In [8]:
years = set(range(core['fiscal_year'].min(), core['fiscal_year'].max() + 1))

missing_years = (core.groupby('club')['fiscal_year'].apply(lambda x: sorted(years - set(x))))

missing_years = missing_years[missing_years.str.len() > 0]

missing_years

club
CSA                        [2025]
Cuiabá                     [2016]
Figueirense          [2024, 2025]
Mirassol                   [2019]
Remo           [2016, 2017, 2018]
Vitória        [2017, 2018, 2019]
Name: fiscal_year, dtype: object

In [9]:
missing_years.str.len().sum()

np.int64(11)

In [10]:
# check the accounting equation allowing small rounding differences
balance_check = core[['club', 'fiscal_year', 'total_assets', 'total_liabilities', 'equity']].copy()

balance_check['difference'] = (
    balance_check['total_assets']
    - balance_check['total_liabilities']
    - balance_check['equity']
)

balance_check[balance_check['difference'].abs() > 1]

,club,fiscal_year,total_assets,total_liabilities,equity,difference


## Keep core variables

Remove helper variables used during the transformation and keep only the variables used in the analytical dataset.

In [11]:
core.drop(labels=['future_results', 'noncurrent_assets_component', 'noncurrent_liabilities'],
          axis=1,
          inplace=True)

core = core[['club', 'fiscal_year', 'cash', 'current_assets', 'total_assets',
             'current_liabilities', 'total_liabilities', 'equity', 'net_result']]

## Export dataset

In [12]:
core.to_csv('../data/processed/core_financials.csv', index=False)